# Chapter 4B — What Makes a Tokenizer Good?
**Series: LM from First Principles** | Prerequisites: Chapter 4A

---

## What this chapter covers

Chapter 4A gave us a working educational BPE tokenizer. After normalizing whitespace,
it represents the corpus using substantially fewer tokens than the character-level representation.

But shorter is not automatically better.

**Which tokenizer gives the model the better representation?**

That question has three parts: what structure did compression destroy, can a model
recover what was lost, and how do we compare tokenizer quality fairly?

The central claim of this chapter:

> **A tokenizer is a contract between the text distribution, the model architecture,
> and the downstream task.**

A good tokenizer should expose task-relevant structure that would otherwise be
difficult for the model to recover, while representing the text compactly.
Polysemy — one word, multiple meanings — is structure the model can recover
from surrounding context; digit alignment in arithmetic is structure that merging
can hide entirely. The chapter explores both.


In [1]:
# input : nothing
# output: text, chars, stoi_char, itos_char, bpe_merges, stoi_bpe, itos_bpe
# Self-contained setup cell -- Chapter 4B runs independently of Chapter 4A.
# Rebuilds the BPE tokenizer from scratch using the fixed regex implementation.
import re, math, urllib.request
from collections import Counter

# ── Load corpus ───────────────────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
with urllib.request.urlopen(url) as f:
    text = f.read().decode('utf-8')

# ── Character tokenizer ───────────────────────────────────────────────────────
chars     = sorted(set(text))
stoi_char = {c: i for i, c in enumerate(chars)}
itos_char = {i: c for i, c in enumerate(chars)}
encode_char = lambda s: [stoi_char[c] for c in s]
decode_char  = lambda ids: ''.join(itos_char[i] for i in ids)
tokens_char  = [stoi_char[c] for c in text]

# ── BPE functions (regex version -- boundary-safe) ───────────────────────────
def build_corpus_vocab(text):
    word_counts = Counter(text.split())
    vocab = {}
    for word, count in word_counts.items():
        vocab[' '.join(list(word) + ['</w>'])] = count
    return vocab

def get_pairs_str(vocab):
    pairs = Counter()
    for word_str, freq in vocab.items():
        syms = word_str.split()
        for i in range(len(syms) - 1):
            pairs[(syms[i], syms[i+1])] += freq
    return pairs

def merge_vocab_str(pair, vocab):
    # Regex ensures we only match at token boundaries (no substring hits inside merged tokens)
    pattern     = r'(?<![^\s])' + re.escape(pair[0]) + r' ' + re.escape(pair[1]) + r'(?![^\s])'
    replacement = pair[0] + pair[1]
    return {re.sub(pattern, replacement, ws): freq for ws, freq in vocab.items()}

def tokenize_word_bpe(word, merges):
    tokens = list(word) + ['</w>']
    for pair, merged in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

def encode_bpe(text_in, merges, stoi):
    ids = []
    for word in text_in.split():
        ids.extend(stoi[t] for t in tokenize_word_bpe(word, merges))
    return ids

def decode_bpe(ids, itos):
    return ''.join(itos[i] for i in ids).replace('</w>', ' ').strip()

# ── Run 500 BPE merges ────────────────────────────────────────────────────────
print("Building BPE vocabulary (500 merges)...")
bpe_vocab  = build_corpus_vocab(text)
bpe_merges = []
for _ in range(500):
    pairs = get_pairs_str(bpe_vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    bpe_merges.append((best, best[0] + best[1]))
    bpe_vocab = merge_vocab_str(best, bpe_vocab)

base_bpe_tokens = {ch for word in text.split() for ch in word}
base_bpe_tokens.add("</w>")
learned_bpe_tokens = {new_token for _, new_token in bpe_merges}
bpe_token_set = base_bpe_tokens | learned_bpe_tokens
bpe_tokens = sorted(bpe_token_set, key=lambda x: (len(x), x))
stoi_bpe   = {t: i for i, t in enumerate(bpe_tokens)}
itos_bpe   = {i: t for i, t in enumerate(bpe_tokens)}

token_count_bpe = sum(len(ws.split()) * freq for ws, freq in bpe_vocab.items())

normalized_text = " ".join(text.split())
print(f"Corpus        : {len(text):,} characters, {len(chars)} unique chars")
print(f"Normalized    : {len(normalized_text):,} characters (whitespace-collapsed)")
print(f"BPE vocab     : {len(bpe_tokens):,} token types  (64 base symbols + 500 learned merges)")
print(f"Token count   : {len(normalized_text):,} chars  ->  {token_count_bpe:,} BPE tokens")
print(f"Compression   : {token_count_bpe/len(normalized_text):.3f} tokens/char  ({len(normalized_text)/token_count_bpe:.1f} chars/token  -- vs normalized text)")


Building BPE vocabulary (500 merges)...


Corpus        : 1,115,394 characters, 65 unique chars
Normalized    : 1,108,152 characters (whitespace-collapsed)
BPE vocab     : 564 token types  (64 base symbols + 500 learned merges)
Token count   : 1,108,152 chars  ->  461,460 BPE tokens
Compression   : 0.416 tokens/char  (2.4 chars/token  -- vs normalized text)


Initialize the scorecard with the compression measurements just computed — the same structure used in Chapter 4A, now filled with values from this run.

In [2]:
# input : token_count_bpe, normalized_text (computed in ch04b-load)
# output: scorecard with all three columns initialised
# Full scorecard for Chapter 4B.
# Compression ratios are derived here rather than hard-coded so that
# 4B always agrees with the actual measured values from 4A's algorithm.

char_compression = 1.0                                     # 1 token per character by definition
bpe_compression  = token_count_bpe / len(normalized_text)  # measured for this run

scorecard = {
    "char_tokenizer": {
        "compression": round(char_compression, 3),
        "boundary_pres": None,  # % morpheme boundaries preserved -- measured in Section 7
        "bpc":           None,  # bits/char -- from research pilot (Section 9)
    },
    "bpe_500": {
        "compression": round(bpe_compression, 3),
        "boundary_pres": None,
        "bpc":         None,
    },
}
import json as _json
print(_json.dumps(scorecard, indent=2))


{
  "char_tokenizer": {
    "compression": 1.0,
    "boundary_pres": null,
    "bpc": null
  },
  "bpe_500": {
    "compression": 0.416,
    "boundary_pres": null,
    "bpc": null
  }
}


---
## Section 7 — What Tokenization Merges and What It Loses

BPE creates a compact representation. But compaction destroys some structure.
Understanding *what* is lost is the key to understanding when tokenizer choice matters.

In [3]:
# input : sentences containing 'bank'  e.g. 'He deposited money at the bank'
# output: 'bank' -> same token ID in all three sentences
# Polysemy: one word, multiple meanings depending on context.
#
# The tokenizer assigns ONE token ID to 'bank' regardless of meaning.
# It only sees the character sequence -- not the surrounding words.
#
# But the surrounding tokens contain enough information to distinguish the senses:
#
#   "He deposited money at the bank"   <- financial context
#   "She sat on the river bank"        <- geographic context
#
# The tokenizer need not resolve this. The information is present in the neighbours.
# We have not yet built the mechanism that lets positions use surrounding information.
# That arrives in Chapter 7.
sentences_bank = [
    "He deposited money at the bank",
    "She sat on the river bank",
    "The bank approved the loan",
]

print("Polysemy: 'bank' gets the same token ID in every context")
print()
for s in sentences_bank:
    toks = []
    for w in s.split():
        toks.extend(tokenize_word_bpe(w, bpe_merges))
    bank_toks = tokenize_word_bpe('bank', bpe_merges)
    bank_ids  = [stoi_bpe[t] for t in bank_toks]
    print(f"  {s!r}")
    print(f"  'bank' -> tokens {bank_toks}  ids {bank_ids}  (identical in all three contexts)")
    print()

print("The distinction between senses is not in the token -- it is in the neighbours.")
print("Chapter 7 will build the mechanism that uses that neighbouring information.")


Polysemy: 'bank' gets the same token ID in every context

  'He deposited money at the bank'
  'bank' -> tokens ['b', 'an', 'k</w>']  ids [38, 96, 315]  (identical in all three contexts)

  'She sat on the river bank'
  'bank' -> tokens ['b', 'an', 'k</w>']  ids [38, 96, 315]  (identical in all three contexts)

  'The bank approved the loan'
  'bank' -> tokens ['b', 'an', 'k</w>']  ids [38, 96, 315]  (identical in all three contexts)

The distinction between senses is not in the token -- it is in the neighbours.
Chapter 7 will build the mechanism that uses that neighbouring information.


### Spot-check: do BPE splits align with morpheme boundaries?

In [4]:
# input : word list  e.g. ['lowest', 'kingly', 'therefore', ...]
# output: BPE splits  e.g. lowest -> [low | est],  whether -> [whe | ther]
# Spot-check how BPE splits words relative to their true morpheme boundaries.
# BPE does not know morphology -- it only counts frequency.
#
# Common words tend to become single tokens because their substrings are common
# enough to be merged early. Rare words may split at arbitrary positions.
#
# Patterns to look for:
#   Clean split:      'lowest'  -> [low | est]   <- both subwords are frequent
#   Frequency-driven: 'because' -> [be | cause]  <- frequency accident, not grammar
#   Arbitrary:        'whether' -> [whe | ther]  <- no morphological meaning
test_words_morphology = [
    "lowest",    # low + est -> clean morpheme split
    "kingly",    # king + ly -> clean
    "therefore", # there + fore -> might split oddly
    "nothing",   # no + thing -> or n + othing?
    "because",   # be + cause -> frequency-driven
    "whether",   # whe + ther -> arbitrary
]

print("Morphology check -- how BPE splits common words:")
print(f"  {'Word':15s}  BPE tokens")
print("  " + "-"*45)
for word in test_words_morphology:
    toks = tokenize_word_bpe(word, bpe_merges)
    display = ' | '.join(t.replace('</w>', '') for t in toks)
    n = len(toks)
    print(f"  {word:15s}  [{display}]  ({n} token{'s' if n>1 else ''})")

print()
print("BPE splits are driven by frequency, not grammar.")
print("A common word often becomes one token; a rare word may split arbitrarily.")
print()
print("Note: this is a spot-check on 6 hand-selected words,")
print("not a population estimate of morphological quality.")


Morphology check -- how BPE splits common words:
  Word             BPE tokens
  ---------------------------------------------
  lowest           [l | ow | est]  (3 tokens)
  kingly           [king | ly]  (2 tokens)
  therefore        [there | fore]  (2 tokens)
  nothing          [no | thing]  (2 tokens)
  because          [be | ca | u | se]  (4 tokens)
  whether          [wh | e | ther]  (3 tokens)

BPE splits are driven by frequency, not grammar.
A common word often becomes one token; a rare word may split arbitrarily.

Note: this is a spot-check on 6 hand-selected words,
not a population estimate of morphological quality.


### Measuring Morpheme-Boundary Preservation

We test words with known morpheme boundaries and check whether every
morpheme boundary is also a BPE token boundary.


In [5]:
# input : words with known morphemes  e.g. 'lowest' -> ['low', 'est']
# output: coherence score  e.g. 7/10 words have all morpheme boundaries in BPE splits
# Formally measure morpheme-boundary preservation.
#
# For each test word we know the correct morpheme boundaries.
# We compute TWO sets of character-position boundaries:
#   morph_bounds: positions where morpheme splits SHOULD occur
#   bpe_bounds  : positions where BPE ACTUALLY splits
#
# A word PRESERVES its morpheme boundaries if every morpheme boundary is also a BPE boundary.
# BPE may add EXTRA splits (more tokens than morphemes) -- that still counts.
# But if BPE MISSES a morpheme boundary, the boundary is not preserved.
#
# Example: 'lowest' = 'low' + 'est', morpheme boundary at character position 3.
#   BPE -> ['low', 'est</w>']  bpe_bounds = {3}  covers morph_bounds {3} -> PRESERVED
#   BPE -> ['lo', 'west</w>']  bpe_bounds = {2}  misses morph_bounds {3} -> not preserved
test_words_morph = {
    "tokenization": ["token", "ize", "ation"],
    "playing":      ["play", "ing"],
    "unhappy":      ["un", "happy"],
    "reformation":  ["re", "form", "ation"],
    "darkness":     ["dark", "ness"],
    "lowest":       ["low", "est"],
    "running":      ["run", "ning"],
    "beautiful":    ["beauti", "ful"],
    "government":   ["govern", "ment"],
    "friendship":   ["friend", "ship"],
}

coherent_count = 0
total_count = 0

print(f"{'Word':<16} {'Morphemes':<28} {'BPE tokens':<28} {'Boundaries preserved?'}")
print("-" * 85)

for word, morphemes in test_words_morph.items():
    bpe_toks  = tokenize_word_bpe(word, bpe_merges)
    bpe_clean = [t.replace('</w>', '') for t in bpe_toks]

    # Build set of character positions where morpheme boundaries occur.
    # 'lowest' with morphemes ['low','est']: boundary after 'low' at pos=3
    morph_bounds = set()
    pos = 0
    for m in morphemes[:-1]:    # skip last -- end-of-word is not a split
        pos += len(m)
        morph_bounds.add(pos)

    # Build set of character positions where BPE splits occur.
    bpe_bounds = set()
    pos = 0
    for t in bpe_clean[:-1]:    # skip last -- same reason
        pos += len(t)
        bpe_bounds.add(pos)

    # Preserved = every morpheme boundary is covered by a BPE boundary
    is_coherent = morph_bounds.issubset(bpe_bounds)
    coherent_count += int(is_coherent)
    total_count += 1

    mark = 'YES' if is_coherent else 'no'
    print(f"  {word:<14} {'+'.join(morphemes):<28} {'|'.join(bpe_clean):<28} {mark}")

pct_coherent = coherent_count / total_count * 100
scorecard["char_tokenizer"]["boundary_pres"] = 100.0  # char tokenizer preserves all boundaries (trivially)
scorecard["bpe_500"]["boundary_pres"] = round(pct_coherent, 1)

print(f'\nMorpheme-boundary preservation: {coherent_count}/{total_count} ({pct_coherent:.0f}%)  [10-word diagnostic set]')
print(f'Char tokenizer: 100% (trivial -- every character boundary is preserved)')
print(f'BPE 500 merges: {pct_coherent:.0f}% boundaries preserved')


Word             Morphemes                    BPE tokens                   Boundaries preserved?
-------------------------------------------------------------------------------------
  tokenization   token+ize+ation              to|k|en|i|z|a|tion           YES
  playing        play+ing                     pla|y|ing                    YES
  unhappy        un+happy                     un|ha|pp|y                   YES
  reformation    re+form+ation                re|for|ma|tion               no
  darkness       dark+ness                    d|ar|kn|ess                  no
  lowest         low+est                      l|ow|est                     YES
  running        run+ning                     r|un|n|ing                   YES
  beautiful      beauti+ful                   b|ea|u|ti|ful                YES
  government     govern+ment                  go|ver|n|m|ent               YES
  friendship     friend+ship                  fri|end|s|hi|p               YES

Morpheme-boundary preservati

**Summary of what BPE preserves vs. loses:**

| Property | BPE | Character | Word |
|---|---|---|---|
| Sequence length | Shorter ✓ | Long ✗ | Very short ✓ |
| Vocabulary size | Medium ✓ | Tiny ✓ | Huge ✗ |
| Unseen words | Decomposable if characters are known ✓ | Decomposable into characters ✓ | Often OOV ✗ |
| Unseen characters | OOV ✗ | OOV ✗ | OOV ✗ |
| Morpheme boundaries | Approximate | Never crosses them | Usually hidden inside word token |
| Digit alignment | May be obscured by merges ✗ | Explicit ✓ | Hidden inside word token ✗ |
| Polysemy | Deferred to contextual model | Deferred | Deferred |

Polysemy is never handled at the tokenizer level — it is always deferred to the model.


---
## Section 8 — The Representation Contract

Here is the central insight of this chapter:

> **A good tokenizer should expose structure that would otherwise be difficult for the
> model to recover, especially when that structure matters to the task.**

The "right" tokenizer is not universal. It depends on what the model can learn and what the task requires.

### Case 1: Polysemy — the contextual model can handle it

```
  "deposit money at the bank"  →  bank  →  token ID 42
  "sit beside the river bank"  →  bank  →  token ID 42

  same initial identity ...

  ... but different surrounding positions
      carry different evidence
```

Both sentences produce the same token ID for "bank".
The distinction between senses is entirely in the surrounding tokens.
A context-aware model can use those neighbours to form different representations.
We will build that mechanism in Chapter 7.
The tokenizer does not need to act here — the information is present, just in the context.

### Case 2: Digit alignment — tokenization can hide useful structure


In [6]:
# input : arithmetic expression  e.g. '1234 + 5678 = 6912'
# output: BPE token list  e.g. ['12', '34</w>', '+</w>', '56', '78</w>', ...]
# Show how BPE tokenizes arithmetic expressions and why digit alignment matters.
#
# Arithmetic requires place-value alignment: to add two numbers, the model must
# compare the units digit of one number to the units digit of the other.
# BPE can fuse digits into subword tokens when digit sequences are frequent
# enough in its training corpus. In that case, individual digit positions
# are no longer explicit token boundaries.
#
# If '1234' -> ['12', '34'] and '5678' -> ['56', '78']:
#   token index 0 covers digit places 1,2 in one number but places 5,6 in another
#   the alignment between place values is hidden inside the token identities
#
# The model can sometimes compensate, but the alignment structure must be inferred
# rather than read directly. Digit-level tokenization exposes it explicitly.
arith_examples = [
    "1234 + 5678 = 6912",
    "99 + 1 = 100",
    "123 + 456 = 579",
]

print("How BPE tokenizes arithmetic expressions:")
print()
for expr in arith_examples:
    toks = []
    for w in expr.split():
        toks.extend(tokenize_word_bpe(w, bpe_merges))
    display = [t.replace('</w>', '') for t in toks if t != '</w>']
    print(f"  {expr!r}")
    print(f"  -> {display}")
    print()

print("Observation: multi-digit numbers may be merged into arbitrary chunks.")
print("'1234' might become ['12', '34'] -- the thousands and hundreds digits are fused.")
print("Digit-level tokenization (one token per digit) preserves place-value alignment:")
for expr in arith_examples:
    print(f"  {expr!r}")
    print(f"  -> {list(expr.replace(' ',''))}")
    print()


How BPE tokenizes arithmetic expressions:

  '1234 + 5678 = 6912'
  -> ['1', '2', '3', '4', '+', '5', '6', '7', '8', '=', '6', '9', '1', '2']

  '99 + 1 = 100'
  -> ['9', '9', '+', '1', '=', '1', '0', '0']

  '123 + 456 = 579'
  -> ['1', '2', '3', '+', '4', '5', '6', '=', '5', '7', '9']

Observation: multi-digit numbers may be merged into arbitrary chunks.
'1234' might become ['12', '34'] -- the thousands and hundreds digits are fused.
Digit-level tokenization (one token per digit) preserves place-value alignment:
  '1234 + 5678 = 6912'
  -> ['1', '2', '3', '4', '+', '5', '6', '7', '8', '=', '6', '9', '1', '2']

  '99 + 1 = 100'
  -> ['9', '9', '+', '1', '=', '1', '0', '0']

  '123 + 456 = 579'
  -> ['1', '2', '3', '+', '4', '5', '6', '=', '5', '7', '9']



### Decision table: when does tokenizer choice matter?

In [7]:
# input : list of (situation, structure_exposed, verdict) cases
# output: decision table  e.g. Polysemy -> tokenizer need not act,  Digit alignment -> tokenizer should act
# Summarise the Representation Contract as a decision table.
# Each row answers: for this task, does the tokenizer choice affect the model's job?
#
# Rule: tokenizer choice matters most when it hides structure that would otherwise
# be difficult for the model to recover from surrounding context.
print("The Representation Contract -- when does tokenizer choice matter?")
print()
cases = [
    ("Polysemy (bank, lead, bark)",
     "Surrounding tokens carry evidence of meaning",
     "Tokenizer choice minor -- model can use context"),
    ("Sequence length",
     "More text per token position = richer context in same budget",
     "BPE improves context coverage"),
    ("Arithmetic digit alignment",
     "Explicit digit boundaries make place-value structure directly available",
     "Digit-level tokenization exposes this; BPE may hide it"),
    ("Rare morphological variants",
     "Shared subword tokens allow generalisation across inflections",
     "BPE fine -- subwords share learned representations"),
    ("Cross-lingual alignment",
     "Same meaning, different scripts",
     "Tokenizer design must account for multiple scripts"),
]
for situation, structure, verdict in cases:
    print(f"  Situation : {situation}")
    print(f"  Structure : {structure}")
    print(f"  Verdict   : {verdict}")
    print()


The Representation Contract -- when does tokenizer choice matter?

  Situation : Polysemy (bank, lead, bark)
  Structure : Surrounding tokens carry evidence of meaning
  Verdict   : Tokenizer choice minor -- model can use context

  Situation : Sequence length
  Structure : More text per token position = richer context in same budget
  Verdict   : BPE improves context coverage

  Situation : Arithmetic digit alignment
  Structure : Explicit digit boundaries make place-value structure directly available
  Verdict   : Digit-level tokenization exposes this; BPE may hide it

  Situation : Rare morphological variants
  Structure : Shared subword tokens allow generalisation across inflections
  Verdict   : BPE fine -- subwords share learned representations

  Situation : Cross-lingual alignment
  Structure : Same meaning, different scripts
  Verdict   : Tokenizer design must account for multiple scripts



**The Representation Contract:** A good tokenizer should expose structure that would
otherwise be difficult for the model to recover, especially when that structure
matters to the task. Everything else can be left to the model.

We have not yet built the mechanism that lets token positions communicate.
That arrives in Chapter 7. For now, note that polysemy resolution draws on evidence
from *other* positions — and digit alignment requires structure *within* a single
position that merging has hidden.


---
## How Do We Compare Two Tokenizers?

We have two tokenizers. We train a model with each and compare.

**First instinct: compare token counts.**

Our BPE(500) tokenizer compresses the normalized corpus substantially below 1 token/character — the scorecard above shows the exact measured value.
BPE wins on compression. But shorter sequences are not always better — what matters
is whether the model *learns well* from them.

**Second instinct: compare validation losses.**

This does not work either. A character model predicts one character at a time —
small steps, concentrated probability, **lower raw loss**. A BPE model predicts one
subword at a time — harder steps, spread over a larger vocabulary, **higher raw loss**.

Higher loss per token ≠ worse model. It means harder predictions.

```
  Model A  char tokenizer   val_loss = 1.80   <- easy steps
  Model B  BPE(500)         val_loss = 2.50   <- harder steps

  Is Model A better?  You cannot tell from raw loss alone.
```

We need a metric that normalises both to the **same unit**:
bits required to encode one character.

The path runs through a simpler question:
how many yes/no questions does a model need to identify the next character?


### Measuring the Character Tokenizer: Surprisal, BPC, and Perplexity

We now have the character tokenizer and the full corpus. We can already calculate three metrics — without training any model.

The idea: use **character frequency** as the probability model. `P('e')` = how often `'e'` appears divided by total characters. This is the **unigram baseline** — no context, just raw frequency.

- **Surprisal** of character x = `-log₂(P(x))` — how many bits to encode this one character  
- **BPC** = average surprisal over all characters in the corpus  
- **Perplexity** = `2^BPC` — the effective number of equally-likely choices per step

This is the weakest possible model for this tokenizer — it knows nothing about context, word structure, or grammar. A useful context-aware language model should improve on this unigram baseline — context and training exist to close this gap.


In [8]:
# input : text corpus  e.g. 1,115,394 characters
# output: BPC = 4.779 bits/char,  Perplexity = 27.46  (unigram baseline, no context)
import math
from collections import Counter

# Step 1: character frequency → unigram probability
char_counts = Counter(text)
total_chars  = len(text)

# Step 2: surprisal per character type  (-log2 of its frequency)
char_surprisal = {
    ch: -math.log2(count / total_chars)
    for ch, count in char_counts.items()
}

# Step 3: BPC = average surprisal over every character in the corpus
bpc_unigram = sum(char_surprisal[ch] for ch in text) / total_chars

# Step 4: perplexity = 2^BPC
perplexity_unigram = 2 ** bpc_unigram

sorted_chars = sorted(char_counts.items(), key=lambda x: -x[1])

print("Per-character surprisal:")
print(f"  {'Char':>6}  {'Count':>8}  {'P(x)':>7}  {'Surprisal':>10}  {'Why'}")
print("  " + "-" * 65)

for ch, count in sorted_chars[:2]:
    p = count / total_chars
    s = char_surprisal[ch]
    print(f"  {repr(ch):>6}  {count:>8,}  {p:>7.4f}  {s:>10.3f} bits  ← very common, low surprisal")

print("  ...")

for ch, count in sorted_chars[-2:]:
    p = count / total_chars
    s = char_surprisal[ch]
    print(f"  {repr(ch):>6}  {count:>8,}  {p:>7.4f}  {s:>10.3f} bits  ← very rare, high surprisal")

print()
print("Averaged over all characters in the corpus:")
print()
print(f"  BPC        : {bpc_unigram:.3f} bits/char")
print(f"  Perplexity : {perplexity_unigram:.2f}   (model behaves as if choosing among ~{perplexity_unigram:.0f} equally likely characters)")
print()
print(f"  Shannon's classic estimates for English      : roughly ~1 bit/char")
print(f"  This unigram baseline (no context)           :  {bpc_unigram:.2f} bits/char")
print(f"  Gap = {bpc_unigram - 1.0:.2f} bits — what context + training recovers")
print()
print(f"  A context-aware model that learns useful predictive structure should beat this unigram baseline.")


Per-character surprisal:
    Char     Count     P(x)   Surprisal  Why
  -----------------------------------------------------------------
     ' '   169,892   0.1523       2.715 bits  ← very common, low surprisal
     'e'    94,611   0.0848       3.559 bits  ← very common, low surprisal
  ...
     '&'         3   0.0000      18.504 bits  ← very rare, high surprisal
     '$'         1   0.0000      20.089 bits  ← very rare, high surprisal

Averaged over all characters in the corpus:

  BPC        : 4.779 bits/char
  Perplexity : 27.46   (model behaves as if choosing among ~27 equally likely characters)

  Shannon's classic estimates for English      : roughly ~1 bit/char
  This unigram baseline (no context)           :  4.78 bits/char
  Gap = 3.78 bits — what context + training recovers

  A context-aware model that learns useful predictive structure should beat this unigram baseline.


> **Recall from Chapter 4A — Bits Per Character (BPC)**  
> BPC is the average number of bits needed to encode one character, given a model's predictions:  
> `BPC = (val_loss / ln(2)) × (n_tokens / n_chars)`  
> The first factor converts loss from nats to bits per token; the second scales to bits per character.  
> This makes BPC comparable across tokenizers — a BPE model's higher raw loss is not a penalty,  
> because each token covers more characters.

---
## Section 9 — Bits Per Character: The Fair Comparison Metric


In [9]:
# input : val_loss + token/char counts  e.g. val_loss=2.50, n_tokens=250000, n_chars=1000000
# output: BPC = 0.9016 bits/char  (lower is better regardless of tokenizer)
import math

def bpc(val_loss, n_tokens, n_chars):
    # Convert val_loss (cross-entropy in nats) to bits per character.
    # Step 1: val_loss / log(2)          -> nats to bits per token
    # Step 2: x (n_tokens / n_chars)     -> bits per token to bits per character
    #   char tokenizer: n_tokens/n_chars = 1.0  (one token per char)
    #   BPE tokenizer:  n_tokens/n_chars ~ 0.414 (~2.4 chars per token on tinyshakespeare)
    return (val_loss / math.log(2)) * (n_tokens / n_chars)


# Same 1M-char corpus, three tokenizers with different raw val_loss values.
# Raw loss is NOT comparable across tokenizers -- BPC normalises by characters covered.
n_chars = 1_000_000

scenarios = [
    ("Character tokenizer",        1.80,  1_000_000),  # 1 token per char
    ("BPE (500 merges, ~2.4ch/tok)", 2.50,   460_000),  # ~2.4 chars per token
    ("BPE (5K merges, ~6ch/tok)",  3.10,    166_000),  # ~6 chars per token
]

print("BPC comparison (hypothetical, 1M-char corpus):")
print(f"  {'Tokenizer':35s}  {'val_loss':>9}  {'tok/char':>8}  {'BPC':>7}")
print("  " + "-"*65)
for name, loss, n_tok in scenarios:
    b = bpc(loss, n_tok, n_chars)
    ratio = n_tok / n_chars
    print(f"  {name:35s}  {loss:>9.2f}  {ratio:>8.4f}  {b:>7.4f}")

print()
print("Why our pilot's difficulty-informed tokenizers all failed:")
print("  Merging tokens reduces n_tokens -> lowers the (n_tok/n_char) factor")
print("  But harder tokens -> higher val_loss -> raises the (loss/ln2) factor")
print("  For an ideal probability model under a lossless re-tokenization, the underlying")
print("  information content is unchanged -- same text, same entropy.")
print("  But a real finite model is not ideal: tokenization changes what regularities")
print("  the model must learn, how context is allocated, and how embeddings generalise.")
print("  So different tokenizers can produce different trained-model BPC in practice.")


BPC comparison (hypothetical, 1M-char corpus):
  Tokenizer                             val_loss  tok/char      BPC
  -----------------------------------------------------------------
  Character tokenizer                       1.80    1.0000   2.5969
  BPE (500 merges, ~2.4ch/tok)              2.50    0.4600   1.6591
  BPE (5K merges, ~6ch/tok)                 3.10    0.1660   0.7424

Why our pilot's difficulty-informed tokenizers all failed:
  Merging tokens reduces n_tokens -> lowers the (n_tok/n_char) factor
  But harder tokens -> higher val_loss -> raises the (loss/ln2) factor
  For an ideal probability model under a lossless re-tokenization, the underlying
  information content is unchanged -- same text, same entropy.
  But a real finite model is not ideal: tokenization changes what regularities
  the model must learn, how context is allocated, and how embeddings generalise.
  So different tokenizers can produce different trained-model BPC in practice.


### Complete the scorecard — fill in the BPC column

In [10]:
# input : BPC pilot values  e.g. char_bpc=3.03, bpe_bpc=2.08
# output: completed 3-row scorecard (compression | morphology | BPC)
# Fill in Measurement 3 in the scorecard: BPC.
# Values come from a research pilot where both tokenizers were trained on tinyshakespeare
# under identical conditions and evaluated with: BPC = (val_loss / ln(2)) x (n_tokens / n_chars)
# BPC values come from a separate research pilot (not computed in this notebook)
# Both tokenizers were trained on tinyshakespeare under identical conditions.
scorecard["char_tokenizer"]["bpc"] = 3.03
scorecard["bpe_500"]["bpc"] = 2.08

print()
print('=' * 62)
print('FINAL SCORECARD')
print('=' * 62)
print(f"{'Metric':<26} {'Char tokenizer':>17} {'BPE 500':>17}")
print('-' * 62)
c = scorecard['char_tokenizer']
b = scorecard['bpe_500']
print(f"{'Compression (tok/char) [nb]':<26} {str(c['compression']):>17} {str(b['compression']):>17}")
print(f"{'Boundary pres. (%) [nb]':<26} {str(c.get('boundary_pres', c.get('morphology','?')))+'%':>17} {str(b.get('boundary_pres', b.get('morphology','?')))+'%':>17}")
print(f"{'BPC (lower = better) [pilot]':<26} {str(c['bpc']):>17} {str(b['bpc']):>17}")
print('=' * 62)
print()
print('BPE wins on compression and BPC.')
print('BPE may lose on morpheme-boundary preservation.')
print('A context-aware model may compensate for some incoherent splits.')



FINAL SCORECARD
Metric                        Char tokenizer           BPE 500
--------------------------------------------------------------
Compression (tok/char) [nb]               1.0             0.416
Boundary pres. (%) [nb]               100.0%             80.0%
BPC (lower = better) [pilot]              3.03              2.08

BPE wins on compression and BPC.
BPE may lose on morpheme-boundary preservation.
A context-aware model may compensate for some incoherent splits.


**Why tokenizer improvements are hard to measure under standard LM training:**

BPC normalises by characters, not tokens. A lossless re-tokenisation that packages
the same text into fewer, larger tokens does not change the underlying information
the model must predict. For an ideal probability model, BPC would be identical.

But real finite models are not ideal coders. Tokenization changes:
- what regularities the model must learn (morpheme boundaries vs. arbitrary cuts)
- how context is distributed (more characters per position = richer each step)
- how embeddings must generalise (shared subwords vs. unique word tokens)

**So different tokenizers can produce measurably different trained-model BPC.**

A tokenizer that genuinely helps must change the *representation the model learns from*,
not just repackage the same information into different-sized chunks.
Digit alignment does this by making place-value structure explicit.
A tokenizer that merely changes compression without changing what regularities
the model can exploit will show little or no BPC improvement.


---
## Section 10 — Real Tokenizers: tiktoken and SentencePiece

Production tokenizers are trained on orders of magnitude more data with vocabulary sizes of 32K–100K.
Production tokenizers use the same subword principle we just built, but at much larger scale and with additional engineering choices: byte-level base vocabularies, alternative training algorithms (Unigram LM, WordPiece), and explicit whitespace handling.

**tiktoken** (OpenAI): used by GPT-3.5, GPT-4, and the GPT family.
- `cl100k_base`: 100,277 tokens. Used by GPT-4.
- `o200k_base`: 200,019 tokens. Used by GPT-4o.

**SentencePiece** (Google): used by many open models including early LLaMA generations, Mistral, PaLM, and others.
- Language-agnostic: handles whitespace as a regular character (no word-boundary assumptions)
- Supports BPE and Unigram LM variants
- Vocabulary sizes vary widely: 32K–128K tokens depending on the model family

The cell below shows tiktoken if installed. If not installed, it shows what the output would look like.


In [11]:
# input : test sentences  e.g. 'To be or not to be, that is the question.'
# output: token counts per tokenizer  e.g. char=33, BPE(500)=12, GPT-4(100K)=10
# Try to use tiktoken; fall back to a static demonstration if not installed
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
    USE_TIKTOKEN = True
    print("tiktoken loaded: cl100k_base (GPT-4 tokenizer)")
except ImportError:
    USE_TIKTOKEN = False
    print("tiktoken not installed. Showing expected output.")
    print("Install with: pip install tiktoken")

test_sentences_tok = [
    "To be or not to be, that is the question.",
    "The bank approved the loan after reviewing the river bank property.",
    "1234 + 5678 = 6912",
    "Tokenization is the first step in any NLP pipeline.",
]

# Expected token counts for the static demo (token ID arrays omitted for safety)
expected_counts = [12, 12, 9, 11]

print()
print(f"  {'Text':55s}  {'# tokens'}")
print("  " + "-"*65)

for i, sentence in enumerate(test_sentences_tok):
    if USE_TIKTOKEN:
        ids = enc.encode(sentence)
        n = len(ids)
    else:
        n = expected_counts[i]
    short = sentence[:52] + "..." if len(sentence) > 52 else sentence
    print(f"  {short:55s}  {n}")

print()
print("Compare to our BPE(500) tokenizer and the char tokenizer:")
sample = "To be or not to be, that is the question."
n_char = len(sample.replace(' ', ''))
n_our_bpe = sum(len(tokenize_word_bpe(w, bpe_merges)) for w in sample.split())
print(f"  Char tokenizer  : {n_char} tokens")
print(f"  Our BPE(500)    : {n_our_bpe} tokens")
if USE_TIKTOKEN:
    n_gpt4 = len(enc.encode(sample))
    print(f"  GPT-4 (100K BPE): {n_gpt4} tokens")


tiktoken loaded: cl100k_base (GPT-4 tokenizer)

  Text                                                     # tokens
  -----------------------------------------------------------------
  To be or not to be, that is the question.                12
  The bank approved the loan after reviewing the river...  12
  1234 + 5678 = 6912                                       10
  Tokenization is the first step in any NLP pipeline.      12

Compare to our BPE(500) tokenizer and the char tokenizer:
  Char tokenizer  : 32 tokens
  Our BPE(500)    : 14 tokens
  GPT-4 (100K BPE): 12 tokens


The key difference between our 564-token vocabulary and GPT-4's 100K vocabulary is scale:
- More merges → longer average token → shorter sequences → more context per forward pass
- But: rarer tokens are seen less often during training → harder to learn good embeddings

Vocabulary size is itself a hyperparameter with a sweet spot, usually between 32K and 128K for large models.


---
## PyTorch Lab — Chapter 4B

Five exercises on tokenizer quality, measurement, and the architecture contract.


In [12]:
# input : bpe_merges list + sample words
# output: table of vocab size and avg tokens/word at 0, 100, 300, 500 merges
# Exercise 1 — BPE compression curve
# Run BPE for 100, 300, 500 merges. At each checkpoint, measure:
# (a) vocabulary size, (b) average tokens per word.
# Print a table and explain the trend.

print("Exercise 1: BPE compression at different merge counts")
print(f"  {'Merges':>8}  {'Vocab size':>12}  {'Avg tok/word':>14}")
print("  " + "-"*40)

# We can estimate from the full bpe_merges list — resimulate by applying only first N merges
# For efficiency, sample a subset of words
sample_words = list(set(text.split()))[:500]

for n_merges in [0, 100, 300, 500]:
    partial_merges = bpe_merges[:n_merges]
    tok_counts = [len(tokenize_word_bpe(w, partial_merges)) for w in sample_words]
    avg = sum(tok_counts) / len(tok_counts)
    partial_tokens = {new_token for _, new_token in bpe_merges[:n_merges]}
    vocab_sz = len(base_bpe_tokens | partial_tokens)
    label = "(char baseline)" if n_merges == 0 else ""
    print(f"  {n_merges:>8}  {vocab_sz:>12,}  {avg:>14.3f}  {label}")


Exercise 1: BPE compression at different merge counts
    Merges    Vocab size    Avg tok/word
  ----------------------------------------


         0            64           8.230  (char baseline)


       100           164           5.616  


       300           364           4.620  


       500           564           4.292  


### Exercise 2 — Most frequent BPE tokens

In [13]:
# input : bpe_vocab (final, 500 merges)
# output: top-15 tokens by frequency  e.g. 'the</w>':5437, 'e</w>':29077
# Exercise 2 — Most common BPE tokens
# After 500 merges, find the 15 most frequent BPE tokens in the corpus.
# What patterns do you see? (common words? common fragments?)

print("Exercise 2: 15 most frequent BPE tokens (500 merges)")
token_freq = Counter()
for word_str, freq in bpe_vocab.items():
    for tok in word_str.split():
        token_freq[tok] += freq

print(f"  {'Rank':>4}  {'Token':>20}  {'Display':>12}  {'Count':>8}")
print("  " + "-"*50)
for rank, (tok, count) in enumerate(token_freq.most_common(15), 1):
    display = tok.replace('</w>', '↵')
    print(f"  {rank:>4}  {tok:>20}  {display:>12}  {count:>8,}")


Exercise 2: 15 most frequent BPE tokens (500 merges)
  Rank                 Token       Display     Count
  --------------------------------------------------
     1                     d             d     5,673
     2                     t             t     5,625
     3               the</w>          the↵     5,473
     4                     s             s     5,333
     5                     e             e     5,093
     6                 I</w>            I↵     4,448
     7                     c             c     4,276
     8                to</w>           to↵     4,165
     9               and</w>          and↵     4,101
    10                     i             i     4,093
    11                 ,</w>            ,↵     4,063
    12                     a             a     4,058
    13                 :</w>            :↵     4,003
    14                 .</w>            .↵     3,977
    15                     b             b     3,943


### Exercise 3 — Arithmetic and digit alignment

In [14]:
# input : arithmetic expression  e.g. '1234 + 5678 = 6912'
# output: list of multi-digit tokens  e.g. ['1234</w>'] -> digits are fused
# Exercise 3 — Arithmetic and digit alignment
# Encode "1234 + 5678 = 6912" with our BPE tokenizer.
# Do any tokens cross digit boundaries? What does this mean for arithmetic tasks?

arith = "1234 + 5678 = 6912"
arith_toks = []
for w in arith.split():
    arith_toks.extend(tokenize_word_bpe(w, bpe_merges))

print("Exercise 3: Arithmetic tokenization")
print(f"  Expression: {arith!r}")
print(f"  BPE tokens: {[t.replace('</w>','') for t in arith_toks if t != '</w>']}")
print()

# Check if any token merges digits
multi_digit = [t for t in arith_toks if sum(c.isdigit() for c in t) > 1]
if multi_digit:
    print(f"  Multi-digit tokens: {multi_digit}")
    print("  → BPE fuses digits. The model cannot directly compare units/tens/hundreds digits.")
    print("  → Digit-level tokenization exposes place-value alignment directly.")
    print("    Multi-digit BPE tokens require the model to infer that structure from context.")
else:
    print("  No multi-digit fusions for this example (our small BPE may not have merged these).")
    print("  With a corpus containing frequent number sequences, additional merges may fuse digits.")


Exercise 3: Arithmetic tokenization
  Expression: '1234 + 5678 = 6912'
  BPE tokens: ['1', '2', '3', '4', '+', '5', '6', '7', '8', '=', '6', '9', '1', '2']

  No multi-digit fusions for this example (our small BPE may not have merged these).
  With a corpus containing frequent number sequences, additional merges may fuse digits.


### Exercise 4 — Three-measurement comparison

In [15]:
# input : two tokenizers with val_loss + token counts
# output: compression ratio, BPC, and chars-per-512-window for each
# Exercise 4 -- All three measurements
# Two tokenizers on the same 1M-char corpus:
#   Tokenizer A: 1,000,000 tokens, val_loss = 2.10
#   Tokenizer B:   420,000 tokens, val_loss = 3.20
# 1. Calculate compression ratio for each.
# 2. Calculate BPC for each.
# 3. Which covers more text in a 512-token window?

n_chars_6 = 1_000_000

ratio_A        = 1_000_000 / n_chars_6
bpc_A          = (2.10 / math.log(2)) * ratio_A
chars_512_A    = int(512 / ratio_A)

ratio_B        = 420_000 / n_chars_6
bpc_B          = (3.20 / math.log(2)) * ratio_B
chars_512_B    = int(512 / ratio_B)

print('Exercise 4: Three-measurement comparison')
print()
print(f"{'Metric':<30} {'Tokenizer A':>14} {'Tokenizer B':>14}")
print('-' * 60)
print(f"{'Compression (tok/char)':<30} {ratio_A:>14.3f} {ratio_B:>14.3f}")
print(f"{'BPC (bits/char)':<30} {bpc_A:>14.4f} {bpc_B:>14.4f}")
print(f"{'Chars per 512-token window':<30} {chars_512_A:>14,} {chars_512_B:>14,}")
print()
winner_bpc = 'A' if bpc_A < bpc_B else 'B'
winner_ctx = 'A' if chars_512_A > chars_512_B else 'B'
print(f'BPC winner          : Tokenizer {winner_bpc}')
print(f'More context/window : Tokenizer {winner_ctx}')
print()
print('Discussion: can a tokenizer win on BOTH BPC and context coverage?')
print('(Hint: what happens to BPC as you increase the number of BPE merges?)')


Exercise 4: Three-measurement comparison

Metric                            Tokenizer A    Tokenizer B
------------------------------------------------------------
Compression (tok/char)                  1.000          0.420
BPC (bits/char)                        3.0297         1.9390
Chars per 512-token window                512          1,219

BPC winner          : Tokenizer B
More context/window : Tokenizer B

Discussion: can a tokenizer win on BOTH BPC and context coverage?
(Hint: what happens to BPC as you increase the number of BPE merges?)


### Exercise 5 — Out-of-vocabulary characters

In [16]:
# input : strings with OOV characters  e.g. 'cafe' (with accent)
# output: list of OOV characters and how production tokenizers handle them
# Exercise 5 — Out-of-vocabulary characters
# Try encoding a string with characters not in the tinyshakespeare training set.
# How does the character tokenizer handle it? How would BPE handle it?

print("Exercise 5: Out-of-vocabulary characters")
print()

oov_examples = [
    "café",       # 'é' not in ASCII-only Shakespeare
    "naïve",      # 'ï' diacritic
    "日本語",      # Japanese characters
    "hello world", # fully in vocab — baseline
]

for s in oov_examples:
    in_vocab = [c for c in s if c in stoi_char]
    oov_chars = [c for c in s if c not in stoi_char]
    print(f"  Input: {s!r}")
    if oov_chars:
        print(f"  OOV characters: {oov_chars}")
        print(f"  → Character tokenizer would CRASH (KeyError) or need a special <unk> token")
        print(f"  → Many production tokenizers use byte-level encoding: represent any char as its UTF-8 bytes")
        for oov_ch in oov_chars:
            byte_values = list(oov_ch.encode("utf-8"))
            hex_repr = "[" + ", ".join(f"0x{b:02X}" for b in byte_values) + "]"
            print(f"    {oov_ch!r} → UTF-8 bytes {hex_repr} → {len(byte_values)} base token(s)")
    else:
        ids = encode_char(s)
        print(f"  All characters in vocab → {len(ids)} tokens")
    print()


Exercise 5: Out-of-vocabulary characters

  Input: 'café'
  OOV characters: ['é']
  → Character tokenizer would CRASH (KeyError) or need a special <unk> token
  → Many production tokenizers use byte-level encoding: represent any char as its UTF-8 bytes
    'é' → UTF-8 bytes [0xC3, 0xA9] → 2 base token(s)

  Input: 'naïve'
  OOV characters: ['ï']
  → Character tokenizer would CRASH (KeyError) or need a special <unk> token
  → Many production tokenizers use byte-level encoding: represent any char as its UTF-8 bytes
    'ï' → UTF-8 bytes [0xC3, 0xAF] → 2 base token(s)

  Input: '日本語'
  OOV characters: ['日', '本', '語']
  → Character tokenizer would CRASH (KeyError) or need a special <unk> token
  → Many production tokenizers use byte-level encoding: represent any char as its UTF-8 bytes
    '日' → UTF-8 bytes [0xE6, 0x97, 0xA5] → 3 base token(s)
    '本' → UTF-8 bytes [0xE6, 0x9C, 0xAC] → 3 base token(s)
    '語' → UTF-8 bytes [0xE8, 0xAA, 0x9E] → 3 base token(s)

  Input: 'hello world'
  All 

---
## Research Connections

**Sennrich, Haddow & Birch (2016) — *Neural Machine Translation of Rare Words with Subword Units***
The paper that introduced BPE to NLP. The original motivation was rare and unknown words in translation — BPE allowed shared subword representations across morphological variants (`walk`, `walks`, `walked` share a `walk` subunit). This chapter built a direct implementation of their Algorithm 1.

**Kudo & Richardson (2018) — *SentencePiece: A simple and language independent subword tokenizer***
SentencePiece treats whitespace as a regular character (represented as `▁`), making it language-agnostic. It supports both BPE and Unigram Language Model tokenization. Most open-weight LLMs (LLaMA, Mistral, Gemma) use SentencePiece. The Unigram variant learns a probabilistic segmentation rather than a deterministic merge sequence.

**Charton (2022) — *Linear Algebra with Transformers***
Studies how transformers learn linear-algebra operations (eigenvalues, linear systems) and finds that the representation of numbers — specifically whether individual components remain individually addressable — substantially affects what the model can learn. This motivates the digit-alignment argument in Section 8: when BPE fuses digits, place-value structure must be inferred rather than read directly. For the specific claim that digit-level tokenization improves arithmetic on addition tasks, see also **Singh & Strouse (2024) — *Tokenization matters for arithmetic***, which directly measures the effect of number tokenization on multi-digit addition accuracy.

**Bassan (2026) — *Which Characters Need Context?* (Paper 1, this series)**
Measured character-specific context gain in natural language and source code. Structural characters (punctuation) have steeper context-gain curves than lexical characters. This opens the Chapter 4 research question: when characters with high context gain are absorbed into subword tokens, is that context gain preserved, weakened, or transformed? If punctuation context gain is absorbed into BPE tokens that span punctuation, the model may lose a signal that matters for structure prediction.


---
## What This Chapter Deliberately Does Not Explain

**WordPiece and Unigram LM tokenization.**
BERT uses WordPiece; SentencePiece supports BPE and Unigram LM tokenization. Like BPE, these approaches construct subword representations, but they differ in how their vocabularies and segmentations are learned.

**Byte-level BPE (GPT-2 and beyond).**
GPT-2 applied BPE directly to UTF-8 bytes, ensuring every possible input is representable with no `<unk>` token. The base vocabulary is 256 bytes rather than a character set. Full coverage at the cost of slightly longer token sequences for non-Latin scripts. Chapter 4 builds character-level BPE; byte-level is the same algorithm on a different base vocabulary.

**SentencePiece training internals.**
The EM training for Unigram LM, the `hard_vocab_limit`, `shrinking_factor`, and `character_coverage` parameters. These are engineering details for production deployments.

**Multilingual tokenization and script handling.**
Latin-script BPE underrepresents languages like Chinese, Japanese, or Arabic, where single characters carry full word meaning. Dedicated multilingual tokenizers (mBERT, XLM-R) handle this. Out of scope here.

**Special tokens.**
`[CLS]`, `[SEP]`, `[PAD]`, `<|endoftext|>`, `<s>`, `</s>` — control tokens injected by the model pipeline, not the text. Their role arrives in the chapter covering fine-tuning and the full inference pipeline.

---
*Next: Chapter 5 — Sampling: Choosing from a Probability Distribution.*
*The model now produces a distribution over all tokens. How do we pick the next one?*
